In [3]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# 座標データのロード
x_coords = np.load("BasicdataforGNN/x_2layer.npy")[:3654]
y_coords = np.load("BasicdataforGNN/y_2layer.npy")[:3654]
z_coords = np.load("BasicdataforGNN/z_2layer.npy")[:3654]

# エッジ情報を読み込む
edges = np.load("BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long)


# データフォルダ
standardized_data_folder = "Defect_4x4_Standardized1"
label_data_folder = "DefectLabels_4x4_test1"

# データとラベルのペアを取得する関数
def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)  # 欠陥なしデータの場合
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"無効なファイル名の形式: {file_name}")
        return None

# データとラベルファイルを対応付け
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Standardized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]

# 欠陥なしのファイルを追加
data_files.append("0Standardized1_nodefect_4x4_ElNOD.npy")
label_files.append("0DefectLabel_nodefect1.npy")

# データとラベルのペアを作成
data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# 有効なペアのみ取得
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]

# ランダムに1000個のペアをサンプリング
num_samples = 1000
random_indices = np.random.choice(len(valid_pairs), num_samples, replace=False)

# トレーニングデータと残りのデータを分ける
train_pairs = [valid_pairs[i] for i in random_indices]
remaining_pairs = [valid_pairs[i] for i in range(len(valid_pairs)) if i not in random_indices]

# 残りのデータを1:1でバリデーションとテストに分割
val_pairs, test_pairs = train_test_split(remaining_pairs, test_size=0.5, random_state=42)

# データ準備の関数
def prepare_data(pairs):
    sampled_data = []
    sampled_labels = []
    
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # データとラベルを読み込む
        values = np.load(data_file_path)[:3654]
        label = np.load(label_file_path)[:3654]
        
        # 座標データと応力データを結合してノード特徴量を作成
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)
    
    # データを結合しテンソルに変換
    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float)
    y = torch.tensor(sampled_labels, dtype=torch.float)  # 回帰のため

    return x, y

# 各データセットを準備
train_x, train_y = prepare_data(train_pairs)
val_x, val_y = prepare_data(val_pairs)
test_x, test_y = prepare_data(test_pairs)

# Dataオブジェクトにまとめる
train_data = Data(x=train_x, edge_index=edge_index, y=train_y)
val_data = Data(x=val_x, edge_index=edge_index, y=val_y)
test_data = Data(x=test_x, edge_index=edge_index, y=test_y)

# GCNモデル
class GCNModel(torch.nn.Module):
    def __init__(self):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(4, 16)
        self.conv2 = GCNConv(16, 32)
        self.conv3 = GCNConv(32, 16)
        self.fc = torch.nn.Linear(16, 1)  # 回帰タスクのため出力は1次元

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        return self.fc(x)

# モデル、損失関数、オプティマイザの定義
model = GCNModel()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.MSELoss()

# データローダーの作成
train_loader = DataLoader([train_data], batch_size=32, shuffle=True)
val_loader = DataLoader([val_data], batch_size=32, shuffle=False)
test_loader = DataLoader([test_data], batch_size=32, shuffle=False)

# 学習と検証
train_losses = []
val_losses = []
test_losses = []

for epoch in range(1, 101):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    train_losses.append(total_loss / len(train_loader))

    # 検証
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch)
            loss = loss_fn(out, batch.y.view(-1, 1))
            val_loss += loss.item()
    
    val_losses.append(val_loss / len(val_loader))
    print(f'Epoch {epoch}, Loss: {train_losses[-1]:.4f}, Validation Loss: {val_losses[-1]:.4f}')

# テストデータでの評価
test_loss = 0
with torch.no_grad():
    for batch in test_loader:
        out = model(batch)
        loss = loss_fn(out, batch.y.view(-1, 1))
        test_loss += loss.item()
test_loss /= len(test_loader)
print(f'Test Loss: {test_loss:.4f}')

plt.figure(figsize=(12, 6))
plt.plot(epochs, train_losses, label='Training Loss', color='blue', marker='o', linestyle='-', markersize=5)
plt.plot(epochs, val_losses, label='Validation Loss', color='orange', marker='s', linestyle='--', markersize=5)
plt.title('Training and Validation Loss over Epochs (Log Scale)', fontsize=16)
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.yscale('log')  # Y軸を対数スケールに設定
plt.xticks(range(0, 101, 10), fontsize=10)  # 10刻みの目盛り
plt.yticks(fontsize=10)
plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()  # レイアウトを自動調整
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(epochs, train_losses, label='Training Loss', color='blue', marker='o', linestyle='-', markersize=5)
plt.plot(epochs, val_losses, label='Validation Loss', color='orange', marker='s', linestyle='--', markersize=5)
plt.title('Training and Validation Loss over Epochs', fontsize=16)
plt.xlabel('Epochs', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.xticks(range(0, 101, 10), fontsize=10)  # 10刻みの目盛り
plt.yticks(fontsize=10)
plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()  # レイアウトを自動調整
plt.show()

# テストロスを出力
print(f'Test Loss: {test_loss:.4f}')


FileNotFoundError: [Errno 2] No such file or directory: 'BasicdataforGNN/x_2layer.npy'